# Running the pipeline locally

An illustration, not new machinery. Every cell calls a function that already exists
in `src/spikesorting/`; the notebook just shows the order and lets you look at
intermediate results, which the CLI scripts do not.

The command line equivalent of the whole thing:

```bash
python scripts/run_pipeline.py --config configs/demo.yaml
```

Use the scripts for real runs (reproducible, loggable, exit codes for batch jobs) and
this notebook when you want to see inside a stage.

**What runs where.** Sync extraction, alignment, export and plotting need only NumPy
and SciPy, so they run on any machine. The sorting cell needs the `kilosort4`
environment with a CUDA build of torch; it is skipped cleanly if that is absent.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from spikesorting import pipeline
from spikesorting.config import load_session

print(REPO)

## 1. Load the session

Two config layers: the session says *what was recorded* (portable), the machine says
*where this computer keeps things*. Same session file runs on the rig, the HPC and here.

`configs/demo.yaml` is an ordinary session config — it points at the demo binary and
sets `skip_sync` / `skip_blackrock` because there is no second system. Swap in your own
session and `machine="windows_rig"` for a real run.

Fetch the demo file first if you have not:
```bash
python scripts/fetch_demo_data.py
```

In [ ]:
config = load_session(REPO / "configs" / "demo.yaml", "mac")
config.paths.mkdirs()

print(f"session:   {config.session}")
print(f"machine:   {config.machine.name}  (device={config.machine.device})")
print(f"outputs:   {config.output_root}")
print(f"cache:     {config.cache_dir}")
print(f"CatGT:     {config.machine.catgt_dir or 'not installed -> NumPy fallback'}")
print(f"TPrime:    {config.machine.tprime_dir or 'not installed -> least-squares fit'}")
print(f"skip_sync={config.skip_sync}  skip_blackrock={config.skip_blackrock}")

problems = config.missing_inputs()
print("\ninputs OK" if not problems else "\n" + "\n".join(f"MISSING: {p}" for p in problems))

## 2. Extract the sync edges

Runs CatGT where available, always runs the NumPy detector, compares them, and writes
the canonical edge files to `<output>/sync/`. Each file is one leading-edge time in
seconds per line — the format CatGT produces and TPrime consumes.

Every stage returns a `StepResult` that reports what it skipped rather than failing.

In [ ]:
result = pipeline.step_extract_sync(config)
print(result.render())

Look at what came out. For a 1 Hz square wave the intervals should sit on 1.000 s with
jitter of about one sample (33 us at 30 kHz).

In [ ]:
for name, edges in sorted(result.data.get("edge_sets", {}).items()):
    times = edges.times_s
    print(f"{name}: {edges.n} edges from {edges.source}, span {edges.span_s:.1f} s")
    if times.size > 2:
        intervals = np.diff(times)
        print(
            f"    interval mean {intervals.mean():.6f} s,"
            f" jitter {intervals.std() * 1e6:.1f} us"
        )
        figure, ax = plt.subplots(figsize=(9, 2.5), dpi=100)
        ax.plot(times[:-1], intervals * 1e3, ".-", markersize=4, linewidth=0.6)
        ax.set_xlabel("time (s)")
        ax.set_ylabel("interval (ms)")
        ax.set_title(name)
        plt.show()

A run with no pulses reports zero edges rather than failing — that is the correct
result for a recording made without SMA1 connected. Check `result.notes`; it also
records the CatGT command that *would* have run, so a rig run can be reproduced by hand.

## 3. Sort with Kilosort4 (step 3)

Needs the `kilosort4` environment and a CUDA GPU. Sorting runs in the machine's SSD
cache and the results are copied out afterwards, so `temp.dat` never touches the
network share.

This is the slow cell. On a real session run it from `scripts/02_sort_neuropixels.py`
instead, so it survives a closed laptop.

In [ ]:
sort_result = pipeline.step_sort_neuropixels(config)
print(sort_result.render())

## 4. Curate in Phy

Not scriptable, and not meant to be:
```bash
conda activate phy
```

```bash
phy template-gui <output>/sorted_neuropixels/params.py
```

Phy writes `cluster_group.tsv`, whose labels override Kilosort's own
`cluster_KSLabel.tsv`. Everything downstream reads the human labels when they exist.

## 5. Align to the Blackrock timebase (steps 8 and 9)

Skipped for the demo — there is no second system. On a real session this is:

1. coarse offset from the 14 s coded bursts (the code resolves *which* 1 Hz cycle);
2. both 1 Hz trains trimmed to their overlapping window;
3. fine alignment via TPrime, or a least-squares fit of the same matched edges;
4. validation against the burst onsets, which were held out of the fit.

In [ ]:
print(pipeline.step_align(config).render())
print(pipeline.step_validate(config).render())

The pure-compute pieces are worth knowing individually — they take arrays and return
arrays, so you can drive them on your own edge lists without any config at all.

Below, two synthetic systems: SpikeGLX starts 137.5 s later on its own clock and runs
30 ppm fast. Note the order — the **coded burst** supplies the coarse offset, and only
then is the 1 Hz train used. Doing it the other way round does not work, for a reason
worth seeing (two cells down).

In [ ]:
from spikesorting.sync import align, burst

OFFSET, DRIFT = 137.5, 30e-6
rng = np.random.default_rng(0)


def coded_bursts(n, interval=14.0, pulses=8, gap=0.02):
    """Dense bursts every `interval` s, each with its own intra-burst pattern."""
    times = []
    for i in range(n):
        onset = i * interval
        times.append(onset)
        times.extend(onset + np.cumsum(gap * (1 + rng.uniform(0, 1, pulses - 1))))
    return np.sort(np.asarray(times))


to_spikeglx = lambda t: (t - OFFSET) / (1 + DRIFT)

blackrock_1hz = np.arange(0.0, 1200.0, 1.0)
blackrock_burst = coded_bursts(int(1200 // 14))
# SpikeGLX started late, so it missed the first 200 s that Blackrock caught.
spikeglx_1hz = to_spikeglx(blackrock_1hz[blackrock_1hz >= 200.0])
spikeglx_burst = to_spikeglx(blackrock_burst[blackrock_burst >= 200.0])

# 1. Coarse offset from the coded burst.
match = burst.match_bursts(blackrock_burst, spikeglx_burst, min_gap_s=7.0, tolerance_s=0.05)
print(f"coarse offset : {match.offset_s:.6f} s  (planted {OFFSET})")
print(f"                {match.n_matched} bursts matched, "
      f"pulse score {match.pulse_score} vs runner-up {match.runner_up_score}")

# 2. Trim both 1 Hz trains to the overlapping window, then 3. fit the fine map.
br_trim, npx_trim = align.trim_pair_to_overlap(
    blackrock_1hz, spikeglx_1hz, match.offset_s, margin_s=0.25
)
ref_idx, other_idx = burst.match_times(br_trim, npx_trim, match.offset_s, tolerance_s=0.25)
mapping = align.fit_linear_map(npx_trim[other_idx], br_trim[ref_idx])

print(f"matched edges : {mapping.n_points}")
print(f"slope         : {mapping.slope:.9f}  ({mapping.drift_ppm:+.2f} ppm)")
print(f"intercept     : {mapping.intercept:.6f} s  (planted {OFFSET})")
print(f"fit residual  : {np.abs(mapping.residuals_s).max() * 1e6:.3f} us max")

### Why the burst has to come first

A 1 Hz square wave is *periodic*, so every whole-second shift fits it equally well.
Asking `estimate_offset` for the offset from the 1 Hz train alone returns the
maximum-overlap alignment, which is off by a whole number of cycles — and because the
subsequent fit is self-consistent with that wrong offset, it looks fine right up until
validation. The coded burst is what breaks the tie.

In [ ]:
naive = burst.estimate_offset(blackrock_1hz, spikeglx_1hz, tolerance_s=0.1)
print(f"from the 1 Hz train alone : {naive:10.3f} s   <- wrong by whole cycles")
print(f"from the coded burst      : {match.offset_s:10.3f} s   <- correct")
print(f"planted                   : {OFFSET:10.3f} s")

### Step 9: validation on held-out bursts

The map was fit on the 1 Hz train, so the burst onsets are held-out data — which is
what makes this a real check rather than a restatement of the fit.

Ignoring the slope is the classic mistake: at 30 ppm the error reaches ~100 ms after an
hour, so an offset-only alignment sails through a quick eyeball and fails step 9.

In [ ]:
br_onsets = burst.group_burst_onsets(blackrock_burst, 7.0)
npx_onsets = burst.group_burst_onsets(spikeglx_burst, 7.0)

offset_only = align.LinearMap(slope=1.0, intercept=OFFSET, n_points=2, residuals_s=np.empty(0))

for label, candidate in [("offset + slope", mapping), ("offset only", offset_only)]:
    report = align.validate_alignment(
        candidate.apply(npx_onsets), br_onsets, tolerance_s=1e-3, match_tolerance_s=1.0
    )
    print(f"{label:>16}: {report.summary()}")

## 6. Export (steps 7 and 10)

Reads the sorted folder (curated or not), computes per-unit metrics, writes figures and
the export bundle. Picks up aligned spike times automatically when they exist and
records which timebase it used in `export_info.json` — check that field rather than
assuming.

In [ ]:
export_result = pipeline.step_export(config, system="neuropixels", groups=("good", "mua"))
print(export_result.render())

In [ ]:
import pandas as pd

units_csv = export_result.data.get("paths", {}).get("units")
if units_csv and Path(units_csv).exists():
    table = pd.read_csv(units_csv)
    print(f"{len(table)} units, timebase = {export_result.data['timebase']}")
    display(table.head(10))
else:
    print("nothing exported yet -- sort first")

## 7. Metrics and plots by hand

Computation and plotting are deliberately separate: `compute_*` returns numbers,
`plot_*` takes already-computed numbers and draws them. That means you can recompute a
metric your own way and still reuse the figures.

In [ ]:
from spikesorting.export import metrics
from spikesorting.export.curated import load_phy_results, select_units
from spikesorting.plots import plot_isi_histogram, plot_mean_waveform

results_dir = config.paths.sorted_np
if (results_dir / "spike_times.npy").exists():
    phy = load_phy_results(results_dir)
    unit_id = int(select_units(phy, ("good",))[0])
    times = phy.times_for(unit_id)

    isi = metrics.compute_isi(times)                       # computation
    counts, edges = metrics.compute_isi_histogram(isi)
    violations = metrics.compute_isi_violations(isi)
    print(f"unit {unit_id}: {times.size} spikes, "
          f"{violations['fraction'] * 100:.2f}% refractory violations")

    figure, axes = plt.subplots(1, 2, figsize=(10, 3.5), dpi=100)
    plot_isi_histogram(counts, edges, ax=axes[0])          # rendering
    from spikesorting.export.final import mean_template_waveform
    waveform, channel = mean_template_waveform(phy, unit_id)
    plot_mean_waveform(waveform, np.arange(waveform.size) / phy.fs * 1e3, ax=axes[1])
    axes[1].set_title(f"channel {channel}")
    figure.tight_layout()
    plt.show()
else:
    print(f"no sorting in {results_dir} -- run the sorting cell first")

## Everything at once

The same stages the CLI runs, in order, honouring the skip flags.

In [ ]:
for name in ["extract_sync", "sort_neuropixels", "sort_blackrock", "align", "validate", "export"]:
    print(pipeline.STEPS[name](config).render())